# Lab 20 — Multi-Agent Research Demo Notebook

Notebook này giúp bạn **thử nghiệm trực tiếp** các khối logic production của bài lab trong `src/`.

**Luồng làm việc:**
1. Khám phá schemas & shared state
2. Khởi tạo LLM + Search service thật
3. Chạy các agent production (Researcher → Analyst → Writer)
4. Supervisor routing + vòng lặp workflow mini
5. Benchmark single-agent vs multi-agent

> Notebook sử dụng OpenRouter cho LLM và Tavily/corpus offline cho tìm kiếm. Kết quả từ provider `mock` sẽ bị chặn.

## 0. Setup

Chạy từ repo root với package đã cài (`pip install -e ".[dev,llm]"`) và file `.env` đã cấu hình.

In [1]:
import sys
from pathlib import Path

# Cho phép import package khi chạy notebook từ thư mục notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

from multi_agent_research_lab.agents import (
    AnalystAgent,
    ResearcherAgent,
    WriterAgent,
)
from multi_agent_research_lab.cli import run_baseline
from multi_agent_research_lab.core.config import get_settings
from multi_agent_research_lab.core.schemas import (
    BenchmarkMetrics,
    ResearchQuery,
)
from multi_agent_research_lab.core.state import ResearchState
from multi_agent_research_lab.evaluation.benchmark import run_benchmark
from multi_agent_research_lab.services.llm_client import LLMClient
from multi_agent_research_lab.services.search_client import SearchClient

print("✅ Import OK — package production sẵn sàng")

✅ Import OK — package production sẵn sàng


## 1. Khám phá Shared State

`ResearchState` là **single source of truth** được truyền qua mọi agent. Mỗi agent đọc state, cập nhật, rồi trả lại.

In [2]:
query = ResearchQuery(
    query="So sánh RAG và fine-tuning cho domain adaptation",
    max_sources=3,
)
state = ResearchState(request=query)

state.record_route("researcher")
state.add_trace_event("demo", {"note": "first route recorded"})

print("Iteration:", state.iteration)
print("Route history:", state.route_history)
print("Trace:", state.trace)

Iteration: 1
Route history: ['researcher']
Trace: [{'name': 'demo', 'payload': {'note': 'first route recorded'}}]


## 2. Real Services

Notebook dùng trực tiếp service production:

- `SearchClient`: Tavily trước, corpus offline nếu Tavily lỗi hoặc rỗng.
- `LLMClient`: gọi model OpenRouter đã cấu hình trong `.env`.
- Provider `mock` bị chặn để notebook không sử dụng dữ liệu giả lập.

In [3]:
settings = get_settings()
search_client = SearchClient(settings=settings)
llm_client = LLMClient(settings=settings)

docs = search_client.search(query.query, max_results=query.max_sources)
if not docs:
    raise RuntimeError("Không tìm được nguồn từ Tavily hoặc corpus offline.")

provider = docs[0].metadata.get("provider", "unknown")
if provider == "mock":
    raise RuntimeError("Notebook không cho phép sử dụng provider mock.")

print("Search provider:", provider)
for document in docs:
    print(f"- {document.title}: {document.snippet[:80]}...")

Search provider: tavily
- RAG vs Fine-Tuning: What's the Difference?: Both RAG and fine-tuning are methods for adapting language models to specific do...
- RAG Vs. Fine Tuning: Which One Should You Choose?: For text generation tasks, fine-tuning often produces more consistent style and ...
- RAG vs Fine-Tuning: Comparison Guide for Enterprise AI | Contextual AI | Contextual AI: Is RAG better than fine-tuning? For enterprise knowledge applications requiring ...


## 3. Production Agents

Mỗi agent tuân theo contract `BaseAgent.run(state) -> state`.

- `ResearcherAgent`: gọi SearchClient và ghi `sources` + `research_notes`.
- `AnalystAgent`: dùng LLM phân tích bằng chứng thành `analysis_notes`.
- `WriterAgent`: dùng LLM viết `final_answer` kèm citation.

In [4]:
researcher = ResearcherAgent(search_client=search_client)
analyst = AnalystAgent(llm_client=llm_client)
writer = WriterAgent(llm_client=llm_client)

agent_state = ResearchState(request=query)
agent_state = researcher.run(agent_state)

if not agent_state.sources:
    raise RuntimeError("Researcher không trả về nguồn.")
if agent_state.sources[0].metadata.get("provider") == "mock":
    raise RuntimeError("Researcher trả về provider mock, notebook dừng.")

print("Agents:", [researcher.name, analyst.name, writer.name])
print(agent_state.research_notes)

Agents: ['researcher', 'analyst', 'writer']
- [1] RAG vs Fine-Tuning: What's the Difference?: Both RAG and fine-tuning are methods for adapting language models to specific domains. Understanding retrieval-augmented generation systems requires understanding the individual components: embeddings, vector databases, retrieval algorithms. Understanding fine-tuning requires understanding large language models, training procedures, and domain adaptation approaches. [...] The distinction matters practically because it affects what happens when your knowledge changes. With RAG, updating knowledge is straightforward: modify documents in the knowledge repository, update embeddings, and immediately the system references current information. With fine-tuning, updating knowledge requires retraining the model, which is expensive and time-consuming. If your domain knowledge evolves rapidly, RAG’s flexibility is a significant advantage. [...] Evaluation approaches differ between RAG and fine-tuning. RA

## 4. Supervisor Routing

Supervisor quyết định agent nào chạy tiếp dựa trên state hiện tại. Policy bên dưới giống luồng production.

In [5]:
MAX_ITERATIONS = settings.max_iterations


def demo_supervisor_route(state: ResearchState) -> str:
    """Trả về một trong: 'researcher' | 'analyst' | 'writer' | 'done'."""
    if state.iteration >= MAX_ITERATIONS or state.errors:
        return "done"
    if not state.sources:
        return "researcher"
    if not state.analysis_notes:
        return "analyst"
    if not state.final_answer:
        return "writer"
    return "done"


print("Initial route:", demo_supervisor_route(ResearchState(request=query)))

Initial route: researcher


## 5. Mini Workflow Loop

Vòng lặp giữ nguyên thứ tự Supervisor → Researcher → Analyst → Writer và ghi lại `route_history`.

In [6]:
def run_demo_workflow(query_text: str) -> ResearchState:
    request = ResearchQuery(query=query_text, max_sources=3)
    workflow_state = ResearchState(request=request)
    agents = {
        "researcher": ResearcherAgent(search_client=search_client),
        "analyst": AnalystAgent(llm_client=llm_client),
        "writer": WriterAgent(llm_client=llm_client),
    }

    while True:
        route = demo_supervisor_route(workflow_state)
        workflow_state.record_route(route)
        if route == "done":
            break
        workflow_state = agents[route].run(workflow_state)

    if (
        workflow_state.sources
        and workflow_state.sources[0].metadata.get("provider") == "mock"
    ):
        raise RuntimeError("Workflow trả về provider mock, notebook dừng.")
    return workflow_state


final_state = run_demo_workflow(
    "So sánh RAG và fine-tuning cho domain adaptation"
)
print("Route history:", final_state.route_history)
print("\n=== FINAL ANSWER ===\n")
print(final_state.final_answer)

Route history: ['researcher', 'analyst', 'writer', 'done']

=== FINAL ANSWER ===

When comparing RAG (Retrieval-Augmented Generation) and fine-tuning for domain adaptation, it's essential to understand their fundamental mechanisms, advantages, and limitations.

### Definition and Mechanism
RAG enhances the generative capabilities of a model by retrieving relevant documents during inference, which adds contextual information to its responses. In contrast, fine-tuning modifies the model's parameters based on specific domain-related training data, allowing it to better understand and generate content relevant to that domain.

### Knowledge Updating
One of RAG's significant advantages is its ability to adapt to new knowledge more easily. By updating the documents in the repository and revising the embeddings, organizations can integrate evolving knowledge without the need for extensive retraining of the model. Fine-tuning, however, necessitates a complete retrain of the model when incorpor

## 6. Benchmark: Single-agent vs Multi-agent

Hai chế độ dùng cùng query, cùng `SearchClient`, cùng model và cùng số nguồn. Baseline tìm kiếm trước rồi gọi LLM đúng một lần.

In [7]:
def run_single_agent(query_text: str) -> ResearchState:
    """Baseline: search một lần và gọi LLM đúng một lần."""
    request = ResearchQuery(query=query_text, max_sources=3)
    result = run_baseline(
        request,
        llm_client=llm_client,
        search_client=search_client,
    )
    if result.sources and result.sources[0].metadata.get("provider") == "mock":
        raise RuntimeError("Baseline trả về provider mock, notebook dừng.")
    return result


def compute_citation_coverage(state: ResearchState) -> float:
    """Tỷ lệ nguồn trong state.sources được nhắc đến trong final_answer."""
    if not state.sources or not state.final_answer:
        return 0.0
    answer = state.final_answer.casefold()
    cited = sum(
        source.title.casefold() in answer
        or (source.url is not None and source.url.casefold() in answer)
        for source in state.sources
    )
    return cited / len(state.sources)


demo_query = "So sánh RAG và fine-tuning cho domain adaptation"
results: list[BenchmarkMetrics] = []

for run_name, runner in [
    ("single_agent", run_single_agent),
    ("multi_agent", run_demo_workflow),
]:
    result_state, metrics = run_benchmark(run_name, demo_query, runner)
    metrics.citation_coverage = compute_citation_coverage(result_state)
    results.append(metrics)

print(f"{'run':<15}{'latency (s)':<15}{'citation cov.':<15}")
for metrics in results:
    print(
        f"{metrics.run_name:<15}"
        f"{metrics.latency_seconds:<15.3f}"
        f"{metrics.citation_coverage!s:<15}"
    )

run            latency (s)    citation cov.  
single_agent   8.459          1.0            
multi_agent    19.166         1.0            


## 7. Next Steps — chuyển sang `src/`

Notebook hiện gọi trực tiếp implementation production:

| Notebook | Đích trong `src/multi_agent_research_lab/` |
|---|---|
| `LLMClient` | `services/llm_client.py` |
| `SearchClient` | `services/search_client.py` |
| `ResearcherAgent` / `AnalystAgent` / `WriterAgent` | `agents/researcher.py`, `agents/analyst.py`, `agents/writer.py` |
| `demo_supervisor_route` | `agents/supervisor.py` |
| `run_demo_workflow` | `graph/workflow.py` |
| `compute_citation_coverage` | `evaluation/benchmark.py` |

Sau khi chạy notebook, verify code chính:

```bash
make lint && make test
python -m multi_agent_research_lab.cli multi-agent --query "When does a multi-agent architecture outperform a single agent?"
bash scripts/check_todos.sh
```